## Prompt-based benchmark (Colab)

Runs a single prompt-based generation with `ExperimentRunner`.

Assumes you executed `00_colab_setup.ipynb` first (deps + `sys.path`).


In [ ]:
from pathlib import Path
import sys

BENCHMARK_ROOT = Path('/content/nav4rails/repositories/FineTuningOnTelecomCluster/benchmark').resolve()
if str(BENCHMARK_ROOT) not in sys.path:
    sys.path.insert(0, str(BENCHMARK_ROOT))

from src.contracts import ExperimentConfig, ModelConfig, PromptConfig
from src.evaluation.runner import ExperimentRunner

mission = "Mission: charger, valider, calculer trajectoire, exécuter, inspection si nécessaire."

cfg = ExperimentConfig(
    name="colab_prompt",
    task="xml_generation",
    output_root=str(BENCHMARK_ROOT / "runs" / "colab_prompt"),
    method="zero_shot",
    model=ModelConfig(model_name_or_path="dummy"),
    prompt=PromptConfig(mode="zero_shot"),
    catalog_path=str(BENCHMARK_ROOT / "data" / "nav4rail_catalog.json"),
    constraints_dir=str(BENCHMARK_ROOT / "constraints"),
)

runner = ExperimentRunner(cfg)

# Replace with a real model call.
# Here we replay a known-good XML for a smoke run.
xml = (BENCHMARK_ROOT / "real_inspection_mission.xml").read_text(encoding="utf-8")
res = runner.run_prompt_experiment(mission=mission, generate_fn=lambda _msgs: xml)
res

In [ ]:
# Inspect artifacts
from pathlib import Path
run_dir = Path(res['run_dir'])
print('run_dir:', run_dir)
print((run_dir/'summary.md').read_text(encoding='utf-8')[:400])
print('--- generated_bt.xml preview ---')
print((run_dir/'generated_bt.xml').read_text(encoding='utf-8')[:400])


In [ ]:
# Few-shot dynamic + schema-guided prompt inspection (no training)
# This cell builds a few-shot pool from the synthetic generator, selects top-k examples,
# and compares prompt sizes (chars/tokens) between modes.

from pathlib import Path
import json

from src.config_loader import load_experiment_config
from src.data.catalog import load_catalog
from src.methods.prompting import render_prompt_bundle

# Build a small few-shot pool (mission+xml) from the synthetic generator
from src.data.synthetic_generator import iter_dataset

pool_path = BENCHMARK_ROOT / "data" / "dataset_synthetic.jsonl"
pool_path.parent.mkdir(parents=True, exist_ok=True)
pool_rows = list(iter_dataset(30))
pool_path.write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in pool_rows) + "\n", encoding="utf-8")
print("few-shot pool:", pool_path)

# Load pool + select examples
from _jsonl import read_jsonl
from _fewshot import load_pool, select_top_k

pool = load_pool(read_jsonl(pool_path))
mission = "Mission: inspection dynamique avec preparation, execution, et recovery."
examples = select_top_k(mission, pool, k=3)

# Compare prompt bundles
cfg_zero = load_experiment_config(BENCHMARK_ROOT / "configs" / "prompt_zero_shot.yaml")
cfg_few = load_experiment_config(BENCHMARK_ROOT / "configs" / "prompt_fewshot_dynamic.yaml")
cfg_schema = load_experiment_config(BENCHMARK_ROOT / "configs" / "prompt_schema_guided.yaml")

catalog = load_catalog(BENCHMARK_ROOT / "data" / "nav4rail_catalog.json")

bundles = {
    "zero_shot": render_prompt_bundle(mission=mission, catalog=catalog, prompt_config=cfg_zero.prompt, xsd_path=cfg_zero.xsd_path),
    "few_shot_dynamic": render_prompt_bundle(
        mission=mission,
        catalog=catalog,
        prompt_config=cfg_few.prompt,
        few_shot_examples=examples,
        xsd_path=cfg_few.xsd_path,
    ),
    "schema_guided": render_prompt_bundle(mission=mission, catalog=catalog, prompt_config=cfg_schema.prompt, xsd_path=cfg_schema.xsd_path),
}

# Estimate token usage if tokenizer is available (downloads tokenizer weights)
try:
    from transformers import AutoTokenizer

    tok = AutoTokenizer.from_pretrained(cfg_zero.model.model_name_or_path, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    def token_count(msgs):
        text = "\n\n".join([m["content"] for m in msgs])
        return len(tok(text).input_ids)

except Exception as exc:
    tok = None
    print("Tokenizer not available:", exc)

for name, b in bundles.items():
    msgs = b["messages"]
    chars = sum(len(m["content"]) for m in msgs)
    approx_tokens = token_count(msgs) if tok is not None else len((msgs[0]["content"] + msgs[1]["content"]).split())
    print("\n---", name, "---")
    print("metadata:", b["metadata"])
    print("chars:", chars, "| approx_tokens:", approx_tokens)
    print("system preview:\n", msgs[0]["content"][:400])
